# OptimLLM Source B labeling on Google Colab

This notebook runs the expensive Source B double-pass labeling stage on a free Google Colab T4 GPU. It uses the free Apache-2.0 `Qwen/Qwen3-8B` model in 4-bit mode and writes artifacts compatible with `scripts/build_phase2_data.py`.

Resolution policy:

- Keep a prompt only when both passes agree on `task_type`.
- If difficulty differs, choose the lower value: `easy < medium < hard`.
- If privacy differs, choose the higher value: `low < medium < high`.

Before running: select **Runtime → Change runtime type → T4 GPU**. From the project, upload `data/.phase2-cache/lmsys-sample.jsonl`. You may also upload `lmsys-label-checkpoint.jsonl` to resume the local run.

In [ ]:
# Install a Qwen3-compatible Transformers version and 4-bit GPU support.
!pip -q install -U "transformers>=4.51.0" accelerate bitsandbytes sentencepiece


In [ ]:
import gc
import hashlib
import json
import os
import re
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path

import torch
from google.colab import drive, files
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "No GPU found. Select Runtime > Change runtime type > T4 GPU."
print(torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")


## Persistent working directory and input files

Artifacts are stored in Google Drive after every batch. A Colab disconnect therefore loses at most one in-flight batch.

In [ ]:
drive.mount("/content/drive")

WORK_DIR = Path("/content/drive/MyDrive/OptimLLM/source_b")
WORK_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PATH = WORK_DIR / "lmsys-sample.jsonl"
CHECKPOINT_PATH = WORK_DIR / "lmsys-label-checkpoint.jsonl"
AGREED_PATH = WORK_DIR / "lmsys-agreed-pre-dedup.jsonl"
FUNNEL_PATH = WORK_DIR / "lmsys-label-funnel.json"

if not SAMPLE_PATH.exists():
    print("Upload data/.phase2-cache/lmsys-sample.jsonl")
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.endswith("lmsys-sample.jsonl")]
    if not candidates:
        raise FileNotFoundError("lmsys-sample.jsonl was not uploaded")
    shutil.copy2(candidates[0], SAMPLE_PATH)

print(f"Input: {SAMPLE_PATH}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print("To import an existing local checkpoint, place lmsys-label-checkpoint.jsonl in this Drive folder before continuing.")


## Configuration

`LIMIT = None` processes every pending prompt. For a short validation run, set it to `20` or `500`. Increase `BATCH_SIZE` only if GPU memory remains stable; 8 is a conservative T4 default.

In [ ]:
MODEL_ID = "Qwen/Qwen3-8B"
BATCH_SIZE = 8
LIMIT = None
MAX_INPUT_TOKENS = 3072
MAX_NEW_TOKENS = 48

TASK_TYPES = {
    "coding", "math", "reasoning", "creative", "summarization",
    "simple_qa", "planning", "data_analysis", "translation",
}
DIFFICULTY_ORDER = ["easy", "medium", "hard"]
PRIVACY_ORDER = ["low", "medium", "high"]

LABEL_PROMPT_A = '''You are a precise classification system. Classify the following user prompt.
Return only valid JSON. No explanation, no markdown, no preamble.

Required keys:
- task_type: coding, math, reasoning, creative, summarization, simple_qa, planning, data_analysis, or translation
- difficulty: easy, medium, or hard
- privacy: low, medium, or high

task_type definitions:
coding — writing, debugging, explaining, or reviewing code
math — arithmetic, algebra, statistics, or mathematical problem solving
reasoning — logic puzzles, argument analysis, causal inference, multi-step deduction
creative — stories, poems, marketing copy, brainstorming, imaginative writing
summarization — condensing or extracting information from provided text
simple_qa — factual lookup, definitions, general knowledge questions
planning — step-by-step plans, project breakdowns, scheduling, strategy
data_analysis — interpreting data, identifying patterns, statistical reasoning
translation — converting between languages

difficulty:
easy — a small language model (1B–4B parameters) can answer correctly
medium — requires a 7B+ model or specialist model to answer well
hard — requires a strong large model; a 7B model would likely fail or be incomplete

privacy:
low — no personal information; safe for any cloud service
medium — professional context, company names, or mildly sensitive role-specific details
high — personal identifiers, credentials, medical, legal, or financial information

Prompt to classify:
<<<PROMPT>>>
{prompt}
<<<END PROMPT>>>'''

LABEL_PROMPT_B = '''Classify this prompt for an AI routing system. Output only JSON, nothing else.
Required format: {{"task_type":"...","difficulty":"...","privacy":"..."}}

task_type: coding | math | reasoning | creative | summarization | simple_qa | planning | data_analysis | translation
difficulty: easy = answerable by 1B–4B; medium = needs 7B specialist or better; hard = needs a strong large model
privacy: low = cloud-safe; medium = business/professional context; high = PII, credentials, medical, legal, or financial content

Prompt:
<<<PROMPT>>>
{prompt}
<<<END PROMPT>>>'''


In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print(f"Loaded {MODEL_ID}")


## Labeling helpers

Each prompt is classified independently in two passes. Thinking mode is disabled to reduce tokens and runtime. Invalid JSON is recorded as `null`; such rows can be retried by rerunning after removing their last checkpoint record, if needed.

In [ ]:
def read_jsonl(path):
    if not Path(path).exists():
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as error:
                    print(f"Skipping malformed line {line_number} in {path}: {error}")
    return rows


def stable_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def parse_label(text):
    text = text.replace("```json", "").replace("```", "").strip()
    match = re.search(r"\{[^{}]*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        value = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    if set(value) != {"task_type", "difficulty", "privacy"}:
        return None
    value = {key: str(item).strip().lower() for key, item in value.items()}
    if value["task_type"] not in TASK_TYPES:
        return None
    if value["difficulty"] not in DIFFICULTY_ORDER:
        return None
    if value["privacy"] not in PRIVACY_ORDER:
        return None
    return value


def render_chat(instruction):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


@torch.inference_mode()
def classify_batch(prompts, template):
    chats = [render_chat(template.format(prompt=prompt)) for prompt in prompts]
    inputs = tokenizer(
        chats,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(model.device)
    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    input_width = inputs["input_ids"].shape[1]
    texts = tokenizer.batch_decode(generated[:, input_width:], skip_special_tokens=True)
    return [parse_label(text) for text in texts], texts


def resolve_labels(label_a, label_b):
    if not label_a or not label_b or label_a["task_type"] != label_b["task_type"]:
        return None
    return {
        "task_type": label_a["task_type"],
        "difficulty": min(
            (label_a["difficulty"], label_b["difficulty"]),
            key=DIFFICULTY_ORDER.index,
        ),
        "privacy": max(
            (label_a["privacy"], label_b["privacy"]),
            key=PRIVACY_ORDER.index,
        ),
    }


def append_checkpoint(records):
    with open(CHECKPOINT_PATH, "a", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


## Sanity check

Run three prompts before starting the full dataset. Confirm that both passes return valid labels.

In [ ]:
sanity_prompts = [
    "Which is larger, 3 or 7?",
    "Write a Python function that merges two sorted lists.",
    "Summarize this confidential legal agreement for client Jane Doe.",
]
sanity_a, raw_a = classify_batch(sanity_prompts, LABEL_PROMPT_A)
sanity_b, raw_b = classify_batch(sanity_prompts, LABEL_PROMPT_B)
for prompt, label_a, label_b in zip(sanity_prompts, sanity_a, sanity_b):
    print(json.dumps({"prompt": prompt, "label_a": label_a, "label_b": label_b}, indent=2))
assert all(sanity_a) and all(sanity_b), f"Invalid sanity output. Raw A={raw_a}; Raw B={raw_b}"


## Run or resume labeling

The progress bar covers only currently pending prompts. Rerunning this cell resumes from the Drive checkpoint.

In [ ]:
sample = read_jsonl(SAMPLE_PATH)
if not sample:
    raise ValueError("The sample file is empty")

completed = {}
for row in read_jsonl(CHECKPOINT_PATH):
    if row.get("prompt_hash"):
        completed[row["prompt_hash"]] = row

pending = []
for row in sample:
    prompt_hash = stable_hash(row["prompt"])
    prior = completed.get(prompt_hash)
    if not prior or prior.get("label_a") is None or prior.get("label_b") is None:
        pending.append(row)

if LIMIT is not None:
    pending = pending[:LIMIT]

print(f"Sample rows: {len(sample):,}")
print(f"Checkpoint hashes: {len(completed):,}")
print(f"Pending this run: {len(pending):,}")

started = time.time()
for start in tqdm(range(0, len(pending), BATCH_SIZE), desc="Double-pass labeling"):
    batch = pending[start:start + BATCH_SIZE]
    prompts = [row["prompt"] for row in batch]
    labels_a, raw_a = classify_batch(prompts, LABEL_PROMPT_A)
    labels_b, raw_b = classify_batch(prompts, LABEL_PROMPT_B)

    records = []
    for row, label_a, label_b, output_a, output_b in zip(batch, labels_a, labels_b, raw_a, raw_b):
        prompt_hash = stable_hash(row["prompt"])
        resolved = resolve_labels(label_a, label_b)
        record = {
            "prompt_hash": prompt_hash,
            "prompt": row["prompt"],
            "cluster_id": row.get("cluster_id"),
            "label_a": label_a,
            "label_b": label_b,
            "exact_agreement": bool(label_a and label_b and label_a == label_b),
            "task_type_agreement": bool(
                label_a and label_b and label_a["task_type"] == label_b["task_type"]
            ),
            "resolved_label": resolved,
            "agreement": resolved is not None,
        }
        if label_a is None:
            record["raw_output_a"] = output_a
        if label_b is None:
            record["raw_output_b"] = output_b
        records.append(record)
        completed[prompt_hash] = record

    append_checkpoint(records)
    del labels_a, labels_b, raw_a, raw_b
    gc.collect()

elapsed = time.time() - started
print(f"Processed {len(pending):,} prompts in {elapsed / 60:.1f} minutes")


## Build the compatible Source B JSON artifacts

In [ ]:
# Reload the checkpoint so this cell also works after reconnecting to Colab.
completed = {}
for row in read_jsonl(CHECKPOINT_PATH):
    if row.get("prompt_hash"):
        completed[row["prompt_hash"]] = row

valid = []
exact_agreements = 0
task_type_agreements = 0
invalid_or_incomplete = 0

with open(AGREED_PATH, "w", encoding="utf-8") as output:
    for row in sample:
        result = completed.get(stable_hash(row["prompt"]))
        if not result:
            continue
        label_a = result.get("label_a")
        label_b = result.get("label_b")
        resolved = resolve_labels(label_a, label_b)
        if not label_a or not label_b:
            invalid_or_incomplete += 1
        if resolved:
            exact = label_a == label_b
            exact_agreements += int(exact)
            task_type_agreements += 1
            accepted = {
                "prompt": row["prompt"],
                **resolved,
                "source": "lmsys_chat_1m",
                "label_agreement": exact,
                "task_type_agreement": True,
                "label_resolution": "lower_difficulty_higher_privacy",
            }
            valid.append(accepted)
            output.write(json.dumps(accepted, ensure_ascii=False) + "\n")

attempted_hashes = {
    stable_hash(row["prompt"])
    for row in sample
    if stable_hash(row["prompt"]) in completed
}
funnel = {
    "sampled_for_labeling": len(sample),
    "label_attempts_completed": len(attempted_hashes),
    "exact_label_agreements": exact_agreements,
    "task_type_agreements": task_type_agreements,
    "accepted_labels": len(valid),
    "rejected_task_disagreement_or_invalid": len(attempted_hashes) - len(valid),
}
with open(FUNNEL_PATH, "w", encoding="utf-8") as handle:
    json.dump(funnel, handle, indent=2)

print(json.dumps(funnel, indent=2))
print("Task type:", dict(Counter(row["task_type"] for row in valid)))
print("Difficulty:", dict(Counter(row["difficulty"] for row in valid)))
print("Privacy:", dict(Counter(row["privacy"] for row in valid)))
print(f"Artifacts saved in {WORK_DIR}")


## Download artifacts for the project

Copy the three extracted files into `data/.phase2-cache/` locally. The checkpoint allows local or future Colab runs to resume; the agreed and funnel files allow the next Source B stage to proceed.

In [ ]:
archive_path = "/content/source-b-colab-artifacts.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (CHECKPOINT_PATH, AGREED_PATH, FUNNEL_PATH):
        archive.write(path, arcname=path.name)
print(archive_path)
files.download(archive_path)
